In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from src.pago_pipeline.ncbi_api import fetch_ncbi_protein_xml_batches
from src.pago_pipeline.ncbi_snapshot import (
    SnapshotMode,
    get_snapshot_manifest_path,
    get_snapshot_protein_uids_path,
    get_snapshot_xml_file_path,
    load_snapshot_manifest,
    resolve_ncbi_protein_uid_snapshot,
    save_ncbi_protein_xml_snapshot,
)
from src.pago_pipeline.storage import sha256_of_file

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================
dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI email and optional API key at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

if not NCBI_EMAIL:
    raise ValueError(
        "NCBI_EMAIL was not found in the environment. "
        "Please define it in your .env file."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NCBI email configured: {bool(NCBI_EMAIL)}")
print(f"NCBI API key configured: {bool(NCBI_API_KEY)}")

Project root: C:\Programming\Python\pAgo-project
NCBI email configured: True
NCBI API key configured: True


In [3]:
# =============================================================================
# CELL 3 — Define configuration
# =============================================================================

SEARCH_QUERY = "PIWI[All Fields] AND Bacteria[Organism]"

UID_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "ncbi" / "protein_uid_snapshots"
)
XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "ncbi" / "protein_xml_snapshots"
)

UID_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
UID_PAGE_SIZE = 1000
UID_MAX_RETRY_ATTEMPTS = 5
UID_REQUEST_DELAY_SECONDS = None

XML_BATCH_SIZE = 100
XML_MAX_RETRY_ATTEMPTS = 5
XML_REQUEST_DELAY_SECONDS = None

UPDATE_LATEST_DIRECTORY = True

print(f"UID snapshot root directory: {UID_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Search query: {SEARCH_QUERY}")

UID snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_uid_snapshots
XML snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots
Search query: PIWI[All Fields] AND Bacteria[Organism]


In [4]:
# =============================================================================
# CELL 4 — Resolve active snapshot
# =============================================================================

uid_snapshot_payload = resolve_ncbi_protein_uid_snapshot(
    snapshot_mode=UID_SNAPSHOT_MODE,
    snapshot_root_directory=UID_SNAPSHOT_ROOT_DIRECTORY,
    search_query=SEARCH_QUERY,
    deduplicate_uids=True,
    sort_uids=True,
    page_size=UID_PAGE_SIZE,
    max_retry_attempts=UID_MAX_RETRY_ATTEMPTS,
    request_delay_seconds=UID_REQUEST_DELAY_SECONDS,
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    update_latest_directory=True,
)

uid_snapshot_directory = uid_snapshot_payload["snapshot_directory"]
uid_snapshot_manifest = uid_snapshot_payload["manifest"]
uid_snapshot_manifest_file_path = uid_snapshot_payload["manifest_file_path"]
protein_uids = uid_snapshot_payload["protein_uids"]

uid_snapshot_manifest_sha256 = sha256_of_file(
    input_file_path=uid_snapshot_manifest_file_path,
)

print(f"Resolved UID snapshot directory: {uid_snapshot_directory}")
print(f"Resolved UID count: {len(protein_uids)}")
print(f"UID snapshot manifest SHA-256: {uid_snapshot_manifest_sha256}")

Latest snapshot is available. Reusing frozen snapshot.
Resolved UID snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_uid_snapshots\latest
Resolved UID count: 41345
UID snapshot manifest SHA-256: d84a2f0183232e5eece4c24c07daffb95df265132e28a4f09a8f03f802b24ee2


In [5]:
# =============================================================================
# CELL 5 — Fetch raw XML batches from NCBI
# =============================================================================

xml_fetch_result = fetch_ncbi_protein_xml_batches(
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    protein_uids=protein_uids,
    batch_size=XML_BATCH_SIZE,
    max_retry_attempts=XML_MAX_RETRY_ATTEMPTS,
    request_delay_seconds=XML_REQUEST_DELAY_SECONDS,
)

print(f"XML batch count: {xml_fetch_result.batch_count}")
print(
    "Normalized UID count used for XML fetch: "
    f"{xml_fetch_result.normalized_protein_uid_count}"
)
print(f"Protein UIDs SHA-256: {xml_fetch_result.protein_uids_sha256}")

Starting XML request for batch 1/414 (attempt 1/5) with 100 protein UIDs.
Received XML response for batch 1/414 in 15.955 seconds (919183 bytes).
NCBI XML request URL: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi
NCBI XML response headers: {'Date': 'Thu, 09 Apr 2026 00:51:04 GMT', 'Server': 'Finatra', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'Content-Security-Policy': 'upgrade-insecure-requests', 'Referrer-Policy': 'origin-when-cross-origin', 'NCBI-SID': '99C2EE30547CCE19_3D15SID', 'NCBI-PHID': '1D334F2B9643A84500002CE5B8173125.1.1.m_7', 'Content-Type': 'text/xml', 'Cache-Control': 'private', 'X-RateLimit-Limit': '10', 'Content-Disposition': 'attachment; filename="sequence.gpx.xml"', 'X-RateLimit-Remaining': '9', 'Access-Control-Allow-Origin': '*', 'Access-Control-Expose-Headers': 'X-RateLimit-Limit,X-RateLimit-Remaining', 'Set-Cookie': 'ncbi_sid=99C2EE30547CCE19_3D15SID; domain=.nih.gov; path=/; expires=Fri, 09 Apr 2027 00:51:03 GMT', '

In [6]:
# =============================================================================
# CELL 6 — Save consolidated XML snapshot
# =============================================================================
xml_snapshot_directory = save_ncbi_protein_xml_snapshot(
    fetch_result=xml_fetch_result,
    snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
    source_uid_snapshot_manifest=uid_snapshot_manifest,
    source_uid_snapshot_manifest_file_path=uid_snapshot_manifest_file_path,
    protein_uids=protein_uids,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)

xml_file_path = get_snapshot_xml_file_path(
    snapshot_directory=xml_snapshot_directory,
)

print(f"Saved XML snapshot directory: {xml_snapshot_directory}")
print(f"Saved consolidated XML file: {xml_file_path}")

Saved XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Saved consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_records.xml


In [7]:
# =============================================================================
# CELL 7 — Print snapshot summary
# =============================================================================
xml_file_sha256 = sha256_of_file(input_file_path=xml_file_path)

print("XML snapshot creation completed successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Consolidated XML SHA-256: {xml_file_sha256}")
print(f"Total protein UIDs covered: {len(protein_uids)}")
print(f"Total XML batches used for consolidation: {xml_fetch_result.batch_count}")

XML snapshot creation completed successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_records.xml
Consolidated XML SHA-256: ada0563932d20f54f68c614df3c40d4fa3520038ad438a41cc471a1eacb970bf
Total protein UIDs covered: 41345
Total XML batches used for consolidation: 414


In [8]:
# =============================================================================
# CELL 8 — Resolve saved snapshot artifacts
# =============================================================================

manifest_file_path = get_snapshot_manifest_path(
    snapshot_directory=xml_snapshot_directory,
)
protein_uids_file_path = get_snapshot_protein_uids_path(
    snapshot_directory=xml_snapshot_directory,
)
xml_snapshot_manifest = load_snapshot_manifest(
    manifest_file_path=manifest_file_path,
)

print(f"Saved XML manifest: {manifest_file_path}")
print(f"Saved protein UIDs file: {protein_uids_file_path}")
print(f"Manifest batch count: {xml_snapshot_manifest['batch_count']}")
print(
    "Manifest consolidated record count: "
    f"{xml_snapshot_manifest['consolidated_record_count']}"
)

Saved XML manifest: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\manifest.json
Saved protein UIDs file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_uids.txt
Manifest batch count: 414
Manifest consolidated record count: 41345


In [9]:
# =============================================================================
# CELL 9 — Print persisted manifest summary
# =============================================================================

saved_manifest_sha256 = sha256_of_file(input_file_path=manifest_file_path)
saved_protein_uids_sha256 = sha256_of_file(input_file_path=protein_uids_file_path)

print("Persisted XML snapshot metadata:")
print(f"Manifest SHA-256: {saved_manifest_sha256}")
print(f"Protein UIDs file SHA-256: {saved_protein_uids_sha256}")
print(f"Manifest XML file name: {xml_snapshot_manifest['xml_file_name']}")
print(f"Manifest XML SHA-256: {xml_snapshot_manifest['xml_file_sha256']}")
print(
    "Manifest immutable snapshot relative path: "
    f"{xml_snapshot_manifest['immutable_snapshot_relative_path']}"
)
print(
    "Manifest source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

Persisted XML snapshot metadata:
Manifest SHA-256: 3bcd96d0ed3e51403b51558d14644f6b8a6c4e191ce34015f7f807f38143f229
Protein UIDs file SHA-256: c0ad9e0d797cc453887cddb14adea255592c18b5edc0319a39a9b012b96b1c09
Manifest XML file name: protein_records.xml
Manifest XML SHA-256: ada0563932d20f54f68c614df3c40d4fa3520038ad438a41cc471a1eacb970bf
Manifest immutable snapshot relative path: snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Manifest source UID snapshot relative path: snapshots\2026-04-06T20-42-12Z__q_891f443d754c


In [10]:
# =============================================================================
# CELL 10 — Inspect first saved batch records
# =============================================================================

first_three_saved_batch_records = xml_snapshot_manifest["batches"][:3]

for saved_batch_record in first_three_saved_batch_records:
    print(f"Batch index: {saved_batch_record['batch_index']}")
    print(
        "Batch UID interval: "
        f"{saved_batch_record['batch_start_index']}"
        f"..{saved_batch_record['batch_end_index']}"
    )
    print(saved_batch_record["xml_payload_sha256"])
    print(saved_batch_record["protein_uid_count"])
    print("---")

Batch index: 1
Batch UID interval: 0..99
c50dc7d313f9a45befdacb9fa9840ace1579b80164b0c4a163a53c50522b0b3d
100
---
Batch index: 2
Batch UID interval: 100..199
c78bb11c8f9579eacffc8da1e22141e53c532c9d2bac2163432121ef9a159406
100
---
Batch index: 3
Batch UID interval: 200..299
f1c8005d85eab5e0d57e17375aec003faa90031fadffd5165abdce64268b1e6a
100
---


In [11]:
# =============================================================================
# CELL 11 — Validate persisted snapshot consistency
# =============================================================================

print("XML snapshot creation completed successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Protein UIDs file: {protein_uids_file_path}")
print(f"Manifest file: {manifest_file_path}")
print(
    "Manifest batch count matches fetch result: "
    f"{xml_snapshot_manifest['batch_count'] == xml_fetch_result.batch_count}"
)
print(
    "Manifest UID SHA-256 matches fetch result: "
    f"{xml_snapshot_manifest['protein_uids_sha256'] == xml_fetch_result.protein_uids_sha256}"
)
print(
    "Manifest UID count matches resolved UID list: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count'] == len(protein_uids)}"
)

XML snapshot creation completed successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_records.xml
Protein UIDs file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\protein_uids.txt
Manifest file: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_xml_snapshots\snapshots\2026-04-09T00-51-02Z__q_891f443d754c\manifest.json
Manifest batch count matches fetch result: True
Manifest UID SHA-256 matches fetch result: True
Manifest UID count matches resolved UID list: True
